# gpt-oss-20b 部署全流程

这个 notebook 覆盖 gpt-oss-20b 的 Transformers、vLLM、SGLang、Ollama、llama.cpp 路线。

gpt-oss 部署最重要的不是只把模型跑起来，而是确认 Harmony、reasoning effort、MXFP4 和后端支持是否正确。

## 1. 推荐顺序

| 排名 | 路线 | 原因 |
| --- | --- | --- |
| 1 | Ollama | `gpt-oss:20b` 本地运行最简单 |
| 2 | vLLM | 服务端 OpenAI-compatible API 优先路线 |
| 3 | Transformers | 验证 tokenizer、模板、输出格式 |
| 4 | SGLang | 看当前版本支持情况 |
| 5 | llama.cpp | 取决于 GGUF 和架构支持 |

## 2. Ollama

Ollama 是本地运行 gpt-oss-20b 的最简单路线。

In [ ]:
print("ollama run gpt-oss:20b")

## 3. vLLM

vLLM 适合把 gpt-oss-20b 部署成服务。必须使用支持 gpt-oss 的 vLLM 版本。

关键参数：

- `--model`：模型名。
- `--max-model-len`：上下文长度。
- `--gpu-memory-utilization`：显存使用比例。
- `--tensor-parallel-size`：多卡张量并行。
- reasoning effort：通过请求参数或模板相关机制控制，取决于后端实现。

In [ ]:
vllm_command = "vllm serve openai/gpt-oss-20b --host 0.0.0.0 --port 8000 --dtype auto --max-model-len 8192 --gpu-memory-utilization 0.90"
print(vllm_command)

## 4. Transformers

Transformers 用于基础验证。它能帮助检查模型能否加载、tokenizer 是否可用、输出是否符合 Harmony 预期。

In [ ]:
# 会下载并加载模型，确认环境后再执行。
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
#
# tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b", trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", torch_dtype="auto", device_map="auto", trust_remote_code=True)
# messages = [{"role": "user", "content": "用三句话解释 MXFP4。"}]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_new_tokens=512)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 5. SGLang 和 llama.cpp

这两条路线必须以当前版本支持为准。不要在支持不完整时写假命令。

- SGLang：如果当前版本支持 gpt-oss，给启动命令和 OpenAI-compatible API 调用。
- llama.cpp：如果 GGUF 转换和运行支持 gpt-oss，再给 GGUF 路线。

In [ ]:
sglang_command = "python -m sglang.launch_server --model-path openai/gpt-oss-20b --host 0.0.0.0 --port 30000 --tp 1 --context-length 8192 --dtype auto"
print(sglang_command)
print("如果当前 SGLang 版本不支持 gpt-oss，应优先使用 vLLM 或 Ollama。")